# Lab 17: Grassmannians and Distances Between Subspaces

This independent-study notebook includes solutions and similar practice questions.

Main idea: compare subspaces by computing principal angles from the singular values of $Q_U^TQ_W$.

## 1. Helper functions

In [ ]:
import numpy as np


def orthonormal_basis(A, tol=1e-12):
    Q, R = np.linalg.qr(A)
    diag = np.abs(np.diag(R))
    r = int(np.sum(diag > tol))
    return Q[:, :r]


def principal_angles(A, B):
    QA = orthonormal_basis(A)
    QB = orthonormal_basis(B)
    s = np.linalg.svd(QA.T @ QB, compute_uv=False)
    s = np.clip(s, 0.0, 1.0)
    return np.arccos(s)


def grassmann_distances(theta):
    return {
        'geodesic': np.linalg.norm(theta),
        'chordal': np.linalg.norm(np.sin(theta)),
        'projection': np.sin(np.max(theta)),
        'Asimov': np.max(theta),
        'Procrustes': 2*np.linalg.norm(np.sin(theta/2)),
        'Fubini-Study': np.arccos(np.prod(np.cos(theta)))
    }

## 2. Worked example: two planes in $\mathbb R^3$

Let $U=\operatorname{span}\{e_1,e_2\}$ and $W=\operatorname{span}\{e_1,\cos(\alpha)e_2+\sin(\alpha)e_3\}$.

In [ ]:
alpha = np.pi/6
U = np.array([[1,0], [0,1], [0,0]], dtype=float)
W = np.array([[1,0], [0,np.cos(alpha)], [0,np.sin(alpha)]], dtype=float)

theta = principal_angles(U, W)
print('Principal angles in radians:', theta)
print('Principal angles in degrees:', theta*180/np.pi)
print(grassmann_distances(theta))

**Answer.** The principal angles are $(0,\alpha)$. For $\alpha=\pi/6$, they are $(0,30^\circ)$.

## 3. Practice with solution: two lines

Let $U=\operatorname{span}\{(1,0)\}$ and $W=\operatorname{span}\{(1,1)\}$. Compute the principal angle.

In [ ]:
U = np.array([[1],[0]], dtype=float)
W = np.array([[1],[1]], dtype=float)
theta = principal_angles(U, W)
print(theta, theta*180/np.pi)

**Solution.** The angle is $\pi/4=45^\circ$, because $(1,1)/\sqrt2$ has dot product $1/\sqrt2$ with $(1,0)$.

## 4. Projection matrices and chordal distance

In [ ]:
def projection_matrix(A):
    Q = orthonormal_basis(A)
    return Q @ Q.T

alpha = 0.7
U = np.array([[1,0], [0,1], [0,0]], dtype=float)
W = np.array([[1,0], [0,np.cos(alpha)], [0,np.sin(alpha)]], dtype=float)
PU = projection_matrix(U)
PW = projection_matrix(W)
theta = principal_angles(U, W)
chordal_from_angles = np.linalg.norm(np.sin(theta))
chordal_from_projections = np.linalg.norm(PU - PW, 'fro')/np.sqrt(2)
print('Chordal from angles:', chordal_from_angles)
print('Chordal from projections:', chordal_from_projections)

**Observation.** The two values agree up to numerical roundoff.

## 5. Similar practice question

Let the principal angles be $0.1,0.4,0.7$. Compute geodesic and chordal distances.

In [ ]:
theta = np.array([0.1, 0.4, 0.7])
print('geodesic =', np.linalg.norm(theta))
print('chordal =', np.linalg.norm(np.sin(theta)))

**Answer.** The geodesic distance is approximately $0.8124$, and the chordal distance is approximately $0.7580$.

## 6. PCA subspace comparison

In [ ]:
np.random.seed(17)
n, m, k = 6, 300, 2
U_true = orthonormal_basis(np.random.randn(n, k))
X = U_true @ np.random.randn(k, m) + 0.1*np.random.randn(n, m)

# Create a related but rotated subspace.
G = np.eye(n)
angle = 0.35
G[2,2] = np.cos(angle); G[2,3] = -np.sin(angle)
G[3,2] = np.sin(angle); G[3,3] = np.cos(angle)
V_true = G @ U_true
Y = V_true @ np.random.randn(k, m) + 0.1*np.random.randn(n, m)

# PCA via SVD.
Ux, sx, _ = np.linalg.svd(X - X.mean(axis=1, keepdims=True), full_matrices=False)
Uy, sy, _ = np.linalg.svd(Y - Y.mean(axis=1, keepdims=True), full_matrices=False)
PX, PY = Ux[:, :k], Uy[:, :k]
theta = principal_angles(PX, PY)
print('PCA subspace angles in degrees:', theta*180/np.pi)
print(grassmann_distances(theta))

## 7. Reflection questions

1. Why is comparing only the first principal component sometimes misleading?
2. Why do we use QR before SVD when the input matrices are not already orthonormal?
3. Which distance seems most interpretable to you: geodesic, chordal, or projection distance?